In [5]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy import stats
import tensorflow as tf
import seaborn as sns
from pylab import rcParams
from sklearn import metrics
from sklearn.model_selection import train_test_split
from scipy import stats
from sklearn.preprocessing import OneHotEncoder

In [ ]:
RANDOM_SEED = 42
columns = ['x-axis', 'y-axis', 'z-axis','activity']
df = pd.read_csv('/SISFALL_5CLASS.csv', header = 0, names = columns)
df = df.dropna()
df

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
df1=df[['x-axis', 'y-axis', 'z-axis']]
df1=df1*(32.0 / 8192.0)
y = df[['activity']]
df1['activity'] = y.values

scale_columns = ['x-axis', 'y-axis', 'z-axis']
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler = scaler.fit(df1[scale_columns])
df1.loc[:, scale_columns] = scaler.transform(df1[scale_columns].to_numpy())
values = df1[['x-axis', 'y-axis', 'z-axis']]
labels = df1['activity']
# X['activity'] = y.values
# values=X[['x-axis', 'y-axis', 'z-axis']]
# labels=X['activity']
values

In [ ]:
def create_dataset(X, y, time_steps=1, step=1):
    Xs, ys = [], []
    for i in range(0, len(X) - time_steps, step):
        v = X.iloc[i:(i + time_steps)].values
        labels = y.iloc[i: i + time_steps]
        Xs.append(v)        
        ys.append(stats.mode(labels)[0][0])
    return np.array(Xs), np.array(ys).reshape(-1, 1)

TIME_STEPS = 200
STEP = 40
##############################  CHECK THISSSSSSSSSSSSSS SAIF!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Xdata, Ylabel = create_dataset(
    values, 
    labels, 
    TIME_STEPS, 
    STEP)

enc = OneHotEncoder(handle_unknown='ignore', sparse=False)
enc = enc.fit(Ylabel)
Ylabel = enc.transform(Ylabel)

X_train, X_test, y_train, y_test = train_test_split(
        Xdata, Ylabel, test_size=0.2, random_state=RANDOM_SEED,stratify=Ylabel)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=RANDOM_SEED)

In [ ]:
(X_train)

# Predict

In [ ]:
from tensorflow.keras.models import load_model

model = load_model('../input/lstmfall/saved_model/1')
from keras.models import load_model
model.save('my_model.h5')  # creates a HDF5 file 'my_model.h5'
model.summary()
def estimate(modelName,inputSignal):
    model = load_model(modelName)
    yhat = model.predict(inputSignal)
    
    indices = []
    for i in range(yhat.shape[0]):
        maxIndex = np.where(yhat[i] == np.amax(yhat[i]))[0]
        indices.append(maxIndex)
        
        print('\nClass of Object Detected for Signal ' + str(i+1) + ' :')
        print('Wall = ' + str(round(yhat[i][0]*100,2)) + '%')
        print('Human = ' + str(round(yhat[i][1]*100,2)) + '%')
        print('Car = ' + str(round(yhat[i][2]*100,2)) + '%')

    y_pred = np.vstack(indices)
    
    return y_pred

In [ ]:
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
y_pred = model.predict(X_test,verbose=1)
#evaluate predictions
acc = accuracy_score(y_test, yhat)
print(acc)

#connect predictions with outputs
for i in range(2):
#     plt.plot(X_train[i])
# #     plt.plot((yhat[i]*100).round())
#     plt.show()
# # 	print(X_test[i], (yhat[i]*100).round())
    print(len(X_train[i]))

In [ ]:
#pip install --upgrade keras

# Build & Train

In [ ]:
from keras.callbacks import ModelCheckpoint
import os
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Flatten,LSTM,Dropout,BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from keras.layers import Bidirectional
from tensorflow.compat.v1.keras.layers import CuDNNLSTM

BATCH_SIZE = 128
model = Sequential()
model.add(BatchNormalization())
model.add(LSTM(256, return_sequences=True, input_shape=([X_train.shape[1], X_train.shape[2]])))
model.add(BatchNormalization())
model.add(LSTM(128, return_sequences=True))
model.add(BatchNormalization())
model.add(LSTM(50))
model.add(Dense(128, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(y_train.shape[1], activation='softmax'))  
# opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['acc'])
history = model.fit(X_train, y_train, epochs=220, batch_size=BATCH_SIZE, validation_data=(X_val,y_val),verbose=1, shuffle=True)

# Save model

In [ ]:
export_dir = 'saved_model/1'

tf.saved_model.save(model, export_dir)
_, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(accuracy)

# Convert to TF lite

In [ ]:
import pathlib
# export_dir = '../input/lstmfall/saved_model/1'


# Convert the model
converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
#                                       tf.lite.OpsSet.SELECT_TF_OPS]
tflite_model = converter.convert()

# Save the model.
with open('model.tflite', 'wb') as f:
  f.write(tflite_model)


tflite_model_file = pathlib.Path('./model.tflite')
tflite_model_file.write_bytes(tflite_model)
    

In [ ]:
# # y_pred=model.predict(X_test)
# import pathlib
# export_dir = '../input/lstmfall/saved_model/1'

# mode = "Speed" 

# if mode == 'Storage':
#     optimization = tf.lite.Optimize.OPTIMIZE_FOR_SIZE
# elif mode == 'Speed':
#     optimization = tf.lite.Optimize.OPTIMIZE_FOR_LATENCY
# else:
#     optimization = tf.lite.Optimize.DEFAULT

# # EXERCISE: Use the TFLiteConverter SavedModel API to initialize the converter

# converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
# # converter.allow_custom_ops = True

# # Set the optimzations
# converter.optimizations = [optimization]
# converter.target_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,tf.lite.OpsSet.SELECT_TF_OPS]
# # Invoke the converter to finally generate the TFLite model
# tflite_model = converter.convert()

# tflite_model_file = pathlib.Path('./model.tflite')
# tflite_model_file.write_bytes(tflite_model)


In [ ]:
# print(tf.__version__)

# Plot

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_cm(y_true, y_pred, class_names):
  cm = confusion_matrix(y_true, y_pred)
  fig, ax = plt.subplots(figsize=(18, 16)) 
  ax = sns.heatmap(
      cm, 
      annot=True, 
      fmt="d", 
      cmap=sns.color_palette("muted"),
      ax=ax)

  plt.ylabel('Actual')
  plt.xlabel('Predicted')
  ax.set_xticklabels(class_names)
  ax.set_yticklabels(class_names)
  b, t = plt.ylim() # discover the values for bottom and top
  b += 0.5 # Add 0.5 to the bottom
  t -= 0.5 # Subtract 0.5 from the top
  plt.ylim(b, t) # update the ylim(bottom, top) values
  #plt.show() # ta-da!
  plt.savefig("confusion_matrix_lstm.png")
plot_cm(enc.inverse_transform(y_test),enc.inverse_transform(y_pred),
        ['D01', 'D02', 'D03', 'D04', 'D05','F01','F02','F03','F04','F05'])

In [ ]:
import matplotlib.pyplot as plt
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(len(loss))
plt.figure(dpi=1200)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()
plt.savefig('loss_val_loss_lstm.png')

In [ ]:
import matplotlib.pyplot as plt
acc = history.history['acc']
val_acc = history.history['val_acc']
epochs = range(len(loss))
plt.figure(dpi=1200)
plt.plot(epochs, acc, 'bo', label='Training Accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation Accuracy')
plt.title('Training and validation Accuracy')
plt.legend()
plt.show()
plt.savefig('acc_val_lstm.png')

In [ ]:

# Load TFLite model and allocate tensors.
# interpreter = tf.lite.Interpreter(model_content=tflite_model)
# interpreter.allocate_tensors()
# input_details = interpreter.get_input_details()
# output_details = interpreter.get_output_details()
# interpreter.set_tensor(input_details[0]['index'], input_data)
# interpreter.invoke()
# # Get input and output tensors.

# output_data = interpreter.get_tensor(output_details[0]['index'])

In [ ]:
# import os
# from tensorflow import keras
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense,Flatten,LSTM,Dropout
# from tensorflow.keras.regularizers import l2
# from tensorflow.keras.optimizers import Adam
# from keras.callbacks import ModelCheckpoint
# import tensorflow as tf

# verbose, epochs, batchSize,nHidden1,nHidden2 = 1, 80, 128,80,30

# checkpoint_path = "training_1/cp.ckpt"
# checkpoint_dir = os.path.dirname(checkpoint_path)

In [ ]:
# import keras
# N_TIME_STEPS, N_FEATURES, n_outputs = X_train.shape[1], X_train.shape[2], y_train.shape[1]
# model = Sequential()
# model.add(LSTM(nHidden1, input_shape=(N_TIME_STEPS,N_FEATURES)))
# model.add(Dropout(0.7))
# model.add(Dense(nHidden2, activation='tanh'))
# model.add(Dense(n_outputs, activation='softmax'))
# model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# # fit network
# # model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=verbose)
# # evaluate model
# # _, accuracy = model.evaluate(X_test, y_train, batch_size=batch_size, verbose=1)
# # print(accuracy)


# cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
#                                                  save_weights_only=True,save_best_only=True,
#                                                  verbose=1)

# # callbacks= [ModelCheckpoint('model.h5', save_weights_only=False, save_best_only=True, verbose=1)]
# history = model.fit(X_train,y_train,shuffle=False,epochs=epochs,verbose=verbose, validation_data=(X_val,y_val), callbacks=cp_callback)

In [ ]:
# _, accuracy = model.evaluate(X_test, y_test, verbose=1)
# print(accuracy)

In [ ]:
# from tensorflow.keras.models import load_model
# model=load_model('./model.h5')
# model.summary()
# model.save("model")
# model=load_model('model')
# loss, acc = model.evaluate(X_test, y_test, verbose=1)
# print("Restored model, accuracy: {:5.2f}%".format(100 * acc))

In [ ]:
# pd.DataFrame(history.history).plot(figsize=(25, 25))
# plt.gca().set_ylim(0, 1) # set the vertical range to [0-1]
# plt.show()

In [ ]:
# from sklearn.metrics import classification_report
# from sklearn.metrics import plot_confusion_matrix
# from sklearn.metrics import confusion_matrix
# y_test_arg=np.argmax(y_test,axis=1)
# Y_pred = np.argmax(model.predict(X_test),axis=1)
# # predictions = model.predict(X_test)
# # cm = confusion_matrix(y_test_arg, Y_pred)
# # plot_confusion_matrix(cm, X_test,np.unique(Y_pred))
# print(confusion_matrix(y_test_arg, Y_pred))
# plot_confusion_matrix(cm, np.argmax(X_test,axis=1),np.unique(Y_pred))

In [ ]:
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix
cm = confusion_matrix(enc.inverse_transform(y_test),enc.inverse_transform(y_pred))

def evaluate_metrics(confusion_matrix, y_test, y_pred, print_result=False, f1_avg='macro'):
    # https://stackoverflow.com/questions/31324218/scikit-learn-how-to-obtain-true-positive-true-negative-false-positive-and-fal
    
    TP = np.diag(confusion_matrix)
    FP = confusion_matrix.sum(axis=0) - TP
    FN = confusion_matrix.sum(axis=1) - TP    
    TN = confusion_matrix.sum() - (FP + FN + TP)
    # Sensitivity, hit rate, recall, or true positive rate
    TPR = TP / (TP + FN)
    # Specificity or true negative rate
    TNR = TN / (TN + FP)
    # Precision or positive predictive value
    PPV = TP / (TP + FP)
    # Negative predictive value
    NPV = TN / (TN + FN)
    # Fall out or false positive rate
    FPR = FP / (FP + TN)
    # False negative rate
    FNR = FN / (TP + FN)
    # False discovery rate
    FDR = FP / (TP + FP)

    # Overall accuracy
    ACC = (TP + TN) / (TP + FP + FN + TN)
    # ACC_micro = (sum(TP) + sum(TN)) / (sum(TP) + sum(FP) + sum(FN) + sum(TN))
    ACC_macro = np.mean(
        ACC)  # to get a sense of effectiveness of our method on the small classes we computed this average (macro-average)

    f1 = f1_score(y_test, y_pred, average=f1_avg)
    kappa = cohen_kappa_score(y_test, y_pred)
    
    if (print_result):
        print("\n")
        print("\n")
        print("============ METRICS ============")
        print(confusion_matrix)
        print("Accuracy (macro) : ", ACC_macro)        
        print("F1 score         : ", f1)
        print("Cohen Kappa score: ", kappa)
        print("======= Per class metrics =======")
        print("Accuracy         : ", ACC)
        print("Sensitivity (TPR): ", TPR)
        print("Specificity (TNR): ", TNR)
        print("Precision (+P)   : ", PPV)
    
    return ACC_macro, ACC, TPR, TNR, PPV, f1, kappa

# acc=evaluate_metrics(cm, enc.inverse_transform(y_pred),enc.inverse_transform(y_test),True)

# 🔮 Prediction on New Sensor Data

Load the trained model and predict activities from IMU sensor CSV files.

In [17]:
def preprocess_sensor_data_sisfall(csv_file):
    """
    Preprocess IMU sensor data for SisFall model prediction.
    
    BNO055 Sensor Preprocessing:
    - 3 features: accelerometer x, y, z only (NO gyroscope)
    - 200 timesteps per window (1 second at 200Hz)
    - NO filtering (training didn't use any filters)
    - NO ADC scaling (BNO055 outputs m/s² directly, not raw ADC like ADXL345)
    - Convert m/s² → g → normalize to (-1, 1)
    """
    import pandas as pd
    from scipy import interpolate
    from sklearn.preprocessing import MinMaxScaler
    
    # Read CSV file
    df = pd.read_csv(csv_file, sep=';')
    print(f"✓ Loaded {len(df)} samples from {csv_file}")
    
    # Extract ONLY accelerometer data (x, y, z)
    acc_data = df[['acc_x', 'acc_y', 'acc_z']].values
    
    # UPSAMPLE from 100Hz to 200Hz using linear interpolation
    print(f"  Original samples: {len(acc_data)} @ 100Hz")
    
    original_time = np.arange(len(acc_data)) * 0.01
    upsampled_time = np.arange(0, original_time[-1], 0.005)
    
    acc_upsampled = np.zeros((len(upsampled_time), 3))
    for i in range(3):
        f = interpolate.interp1d(original_time, acc_data[:, i], kind='linear')
        acc_upsampled[:, i] = f(upsampled_time)
    
    print(f"  Upsampled samples: {len(acc_upsampled)} @ 200Hz")
    
    # ✅ Convert BNO055 m/s² to g-force (NO ADC scaling needed for BNO055)
    acc_g = acc_upsampled / 9.81  # BNO055 outputs m/s², convert to g
    print(f"  Converted to g-force (BNO055 sensor, no ADC scaling)")
    
    # ⚠️ WARNING: Fitting new scaler per file creates inconsistent normalization!
    # Ideally should use the SAME scaler fitted on training data
    # But since we don't have the saved scaler, we fit a new one
    scaler = MinMaxScaler(feature_range=(-1, 1))
    acc_scaled = scaler.fit_transform(acc_g)
    
    print(f"  Applied MinMaxScaler(-1, 1)")
    print(f"  Data range: [{acc_scaled.min():.3f}, {acc_scaled.max():.3f}]")
    print(f"  ⚠️  NOTE: Scaler fitted on this file only (may differ from training scale)")
    
    # Create sliding windows
    TIME_STEPS = 200
    STEP = 40
    
    windows = []
    for i in range(0, len(acc_scaled) - TIME_STEPS, STEP):
        window = acc_scaled[i:i + TIME_STEPS]
        windows.append(window)
    
    windows = np.array(windows, dtype=np.float32)
    
    print(f"✓ Created {len(windows)} windows of shape {windows.shape}")
    
    return windows

print("✓ Preprocessing function defined (NO filtering, BNO055 sensor, matches training logic)")

✓ Preprocessing function defined (NO filtering, BNO055 sensor, matches training logic)


In [1]:
# def preprocess_sensor_data_sisfall(csv_file):
#     """
#     Preprocess IMU sensor data for SisFall model prediction.
    
#     SisFall model expects:
#     - 3 features: accelerometer x, y, z only (NO gyroscope)
#     - 200 timesteps per window (1 second at 200Hz)
#     - Same preprocessing as SisFall paper (Sucerquia et al., 2017)
    
#     Args:
#         csv_file: Path to CSV with format: timestamp;seconds_elapsed;acc_z;acc_y;acc_x;gyro_z;gyro_y;gyro_x
    
#     Returns:
#         windows: np.array of shape (num_windows, 200, 3)
#     """
#     import pandas as pd
#     from scipy import interpolate
#     from scipy.signal import butter, filtfilt
    
#     # Read CSV file
#     df = pd.read_csv(csv_file, sep=';')
#     print(f"✓ Loaded {len(df)} samples from {csv_file}")
    
#     # Extract ONLY accelerometer data (x, y, z)
#     acc_data = df[['acc_x', 'acc_y', 'acc_z']].values / 9.81
    
#     # UPSAMPLE from 100Hz to 200Hz using linear interpolation
#     print(f"  Original samples: {len(acc_data)} @ 100Hz")
    
#     # Create original time indices (100Hz = 0.01s per sample)
#     original_time = np.arange(len(acc_data)) * 0.01
    
#     # Create upsampled time indices (200Hz = 0.005s per sample)
#     upsampled_time = np.arange(0, original_time[-1], 0.005)
    
#     # Interpolate each axis
#     acc_upsampled = np.zeros((len(upsampled_time), 3))
#     for i in range(3):
#         f = interpolate.interp1d(original_time, acc_data[:, i], kind='linear')
#         acc_upsampled[:, i] = f(upsampled_time)
    
#     print(f"  Upsampled samples: {len(acc_upsampled)} @ 200Hz")
    
#     # Apply same preprocessing as SisFall training:
#     # 1. Convert to g-force (same ADC range as ADXL345: ±8g, 13 bits)
#     # acc_upsampled = acc_upsampled * (32.0 / 8192.0)
    
#     # 2. LOW-PASS FILTER: 4th order Butterworth with 5 Hz cutoff
#     # This is EXACTLY what SisFall paper used (Sucerquia et al., 2017, Section 3.4.1)
#     def butter_lowpass_filter(data, cutoff=5, fs=200, order=4):
#         """4th order IIR Butterworth low-pass filter with 5 Hz cutoff"""
#         nyquist = 0.5 * fs
#         normal_cutoff = cutoff / nyquist
#         b, a = butter(order, normal_cutoff, btype='low', analog=False)
#         return filtfilt(b, a, data, axis=0)
    
#     acc_filtered = butter_lowpass_filter(acc_upsampled, cutoff=5, fs=200, order=4)
#     print(f"  Applied 4th order Butterworth low-pass filter (5Hz cutoff)")
    
#     # 3. Normalize to (-1, 1) range using MinMaxScaler
#     from sklearn.preprocessing import MinMaxScaler
#     scaler = MinMaxScaler(feature_range=(-1, 1))
#     acc_scaled = scaler.fit_transform(acc_filtered)
    
#     print(f"  Applied MinMaxScaler(-1, 1)")
#     print(f"  Data range: [{acc_scaled.min():.3f}, {acc_scaled.max():.3f}]")
    
#     # Create sliding windows
#     TIME_STEPS = 200  # 1 second at 200Hz (matching training data)
#     STEP = 40         # 0.2 seconds overlap
    
#     windows = []
#     for i in range(0, len(acc_scaled) - TIME_STEPS, STEP):
#         window = acc_scaled[i:i + TIME_STEPS]
#         windows.append(window)
    
#     windows = np.array(windows, dtype=np.float32)
    
#     print(f"✓ Created {len(windows)} windows")
#     print(f"  Window shape: {windows.shape}")
#     print(f"  Expected: (num_windows, 200, 3)")
#     print(f"  Temporal duration: 1 second per window @ 200Hz")
    
#     return windows

# print("✓ Preprocessing function defined (using SisFall paper filter: 4th order Butterworth @ 5Hz)")

✓ Preprocessing function defined (using SisFall paper filter: 4th order Butterworth @ 5Hz)


In [10]:
def predict_sisfall_activity(csv_file, model_path='saved_model/1'):
    """
    Predict fall/activity from sensor data using trained SisFall model.
    
    Args:
        csv_file: Path to sensor CSV file
        model_path: Path to saved model (default: 'saved_model/1')
    
    Returns:
        activity: Predicted activity class
        confidence: Average confidence percentage
    """
    from tensorflow.keras.models import load_model
    from collections import Counter
    
    #Activity labels (adjust based on your SisFall classes)
    #These are examples - update with your actual class names
    ACTIVITY_LABELS = {
        1: 'D01',
        2: 'D02', 
        3: 'D03',
        4: 'D04',
        5: 'D05',
        6: 'F01',
        7: 'F02',
        8: 'F03',
        9: 'F04',
        10: 'F05'
    }


    # ACTIVITY_LABELS = {
    #     1: 'F01',
    #     2: 'F02', 
    #     3: 'F03',
    #     4: 'F04',
    #     5: 'F05',
    #     6: 'D01',
    #     7: 'D02',
    #     8: 'D03',
    #     9: 'D04',
    #     10: 'D05'
    # }

    
    # Preprocess sensor data
    print("\n" + "="*50)
    print("PREPROCESSING SENSOR DATA")
    print("="*50)
    windows = preprocess_sensor_data_sisfall(csv_file)
    
    # Load trained model
    print("\n" + "="*50)
    print("LOADING MODEL")
    print("="*50)
    model = load_model(model_path)
    print(f"✓ Model loaded from {model_path}")
    
    # Make predictions
    print("\n" + "="*50)
    print("MAKING PREDICTIONS")
    print("="*50)
    predictions = model.predict(windows, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)
    
    # Get most common prediction (voting)
    vote_counts = Counter(predicted_classes)
    most_common_class = vote_counts.most_common(1)[0][0]
    vote_count = vote_counts.most_common(1)[0][1]
    
    # Calculate confidence
    confidences = [predictions[i][predicted_classes[i]] for i in range(len(windows))]
    avg_confidence = np.mean(confidences) * 100
    
    # Get activity name
    activity_name = ACTIVITY_LABELS.get(most_common_class, f"Class_{most_common_class}")
    
    # Print results
    print(f"\n{'='*50}")
    print(f"PREDICTION RESULTS")
    print(f"{'='*50}")
    print(f"\n  Activity: {activity_name}")
    print(f"  Confidence: {avg_confidence:.1f}%")
    print(f"  Windows: {vote_count}/{len(windows)} voted for this class")
    print(f"\n  Vote breakdown:")
    for cls, count in vote_counts.most_common():
        pct = count / len(windows) * 100
        label = ACTIVITY_LABELS.get(cls, f"Class_{cls}")
        print(f"    {label:20s}: {count:3d} ({pct:.1f}%)")
    print(f"{'='*50}\n")
    
    return activity_name, avg_confidence

print("✓ Prediction function defined")

✓ Prediction function defined


In [9]:
from tensorflow.keras.models import load_model

# Load the trained SisFall model (.keras format)
# Note: Model is in parent directory's model folder
model = load_model('../model/model_v011.keras')
model.summary()

print(f"\n✓ Model loaded successfully!")
print(f"Input shape: {model.input_shape}")
print(f"Output shape: {model.output_shape}")
print(f"Number of classes: {model.output_shape[-1]}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_normalization             │ (None, 200, 3)         │            12 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 200, 256)       │       266,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 200, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 200, 128)       │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 200, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 50)             │        35,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         6,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,017,048 (3.88 MB)

 Trainable params: 508,008 (1.94 MB)

 Non-trainable params: 1,030 (4.02 KB)

 Optimizer params: 508,010 (1.94 MB)


✓ Model loaded successfully!
Input shape: (None, 200, 3)
Output shape: (None, 10)
Number of classes: 10


## 🧪 Example: Predict from Your Sensor Data

Run prediction on your IMU sensor CSV file:

In [18]:
cv_path = 'data/sensor_data/Recorded/falling_forward_20260122_193050.csv'

# Example: Predict activity from your sensor data
activity, confidence = predict_sisfall_activity(
    csv_file=f'../{cv_path}',
    model_path='../model/model_v011.keras'  # or 'my_model.h5' if you saved as .h5
)

print(f"\n✓ Final prediction: {activity} ({confidence:.1f}% confidence)")


PREPROCESSING SENSOR DATA
✓ Loaded 480 samples from ../data/sensor_data/Recorded/falling_forward_20260122_193050.csv
  Original samples: 480 @ 100Hz
  Upsampled samples: 958 @ 200Hz
  Converted to g-force (BNO055 sensor, no ADC scaling)
  Applied MinMaxScaler(-1, 1)
  Data range: [-1.000, 1.000]
  ⚠️  NOTE: Scaler fitted on this file only (may differ from training scale)
✓ Created 19 windows of shape (19, 200, 3)

LOADING MODEL


/Users/didiermupenda/miniforge3/envs/dl-projects/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 23 variables whereas the saved optimizer has 44 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


✓ Model loaded from ../model/model_v011.keras

MAKING PREDICTIONS

PREDICTION RESULTS

  Activity: D05
  Confidence: 90.7%
  Windows: 8/19 voted for this class

  Vote breakdown:
    D05                 :   8 (42.1%)
    D03                 :   6 (31.6%)
    F04                 :   4 (21.1%)
    F02                 :   1 (5.3%)


✓ Final prediction: D05 (90.7% confidence)
